# Fine-Tuning GPT-2 for Masked Language Modeling (MLM)
This notebook fine-tunes a GPT-2 model using Masked Language Modeling objectives, randomly masking tokens in the input text and training the model to predict the masked tokens.


## 1. Library Imports & Tokenizer Configuration
We load GPT-2 and configure tokenizer parameters to support sequence-wide padding and token masking.


In [1]:
import torch
from torchtext import data
import spacy
from spacy.symbols import ORTH
from torchtext.datasets import WikiText2

my_tok = spacy.load('en')
 
def spacy_tok(x):
    return [tok.text for tok in my_tok.tokenizer(x)]
 
TEXT = data.Field(lower=True, tokenize=spacy_tok)


In [2]:
my_tok.tokenizer.add_special_case("don't", [{ORTH: "do"}, {ORTH: "n't"}])


In [3]:
train, valid, test = WikiText2.splits(TEXT) 

In [4]:
TEXT.build_vocab(train)

In [5]:
TEXT.vocab.itos = TEXT.vocab.itos + ['[MASK]']
TEXT.vocab.stoi['[MASK]']=len(TEXT.vocab.itos)

In [6]:
batch_size = 50
bptt = 200

In [7]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

cuda


In [8]:
train_iter, valid_iter, test_iter = data.BPTTIterator.splits(
    (train, valid, test),
    batch_size=batch_size,
    bptt_len=bptt, 
    device=device,
    repeat=False)

## 2. Random Token Masking (MLM)
We set up a helper function to randomly replace tokens with `[MASK]` tokens during data pipeline mapping.


In [9]:
mlm_probability = 0.15
vocab_size = len(TEXT.vocab)+1

def mask_tokens(inputs):
    labels = inputs.clone()
    masked_indices = torch.bernoulli(torch.full(labels.shape, mlm_probability)).bool()
    labels[~masked_indices] = -1                            # We only compute loss on masked tokens
    indices_replaced = torch.bernoulli(torch.full(labels.shape, 0.8)).bool() & masked_indices
    inputs[indices_replaced] = TEXT.vocab.stoi['[MASK]']    # 80% of the time, replace masked input tokens with [MASK]
    indices_random = torch.bernoulli(torch.full(labels.shape, 0.5)).bool() & masked_indices & ~indices_replaced
    random_words = torch.randint(vocab_size, labels.shape, dtype=torch.long, device=device)
    inputs[indices_random] = random_words[indices_random]   # 10% of the time, replace masked input tokens with random word
    return inputs, labels

In [10]:
import torch
import torch.nn as nn
import numpy as np

def create_sinusoidal_embeddings(embeds):
    position_enc = torch.tensor([
        [pos / np.power(10000, 2 * (j // 2) / embeds.embedding_dim) for j in range(embeds.embedding_dim)]
                                                                    for pos in range(embeds.num_embeddings)])
    embeds.weight[:, 0::2] = torch.sin(position_enc[:, 0::2])
    embeds.weight[:, 1::2] = torch.cos(position_enc[:, 1::2])
    embeds.weight.detach_()
    embeds.weight.requires_grad = False


class Transformer(nn.Module):
    def __init__(self, embed_dim, hidden_dim, num_embeddings, num_max_positions, num_heads, num_layers, dropout,
                 sinusoidal_embeddings, causal=False):
        """ Transformer (GPT-2 architecture) """
        super().__init__()
        self.causal = causal
        self.tokens_embeddings = nn.Embedding(num_embeddings, embed_dim)
        self.position_embeddings = nn.Embedding(num_max_positions, embed_dim)
        if sinusoidal_embeddings:
            create_sinusoidal_embeddings(self.position_embeddings)
        self.dropout = nn.Dropout(dropout)

        self.attentions, self.feed_forwards = nn.ModuleList(), nn.ModuleList()
        self.layer_norms_1, self.layer_norms_2 = nn.ModuleList(), nn.ModuleList()
        for _ in range(num_layers):
            self.attentions.append(nn.MultiheadAttention(embed_dim, num_heads, dropout = dropout))
            self.feed_forwards.append(nn.Sequential(nn.Linear(embed_dim, hidden_dim),
                                                    nn.ReLU(),
                                                    nn.Linear(hidden_dim, embed_dim)))
            self.layer_norms_1.append(nn.LayerNorm(embed_dim, eps=1e-12))
            self.layer_norms_2.append(nn.LayerNorm(embed_dim, eps=1e-12))

    def forward(self, x, padding_mask = None):
        """ Input has shape [seq length, batch] """
        positions = torch.arange(len(x), device=x.device).unsqueeze(-1)
        h = self.tokens_embeddings(x)
        h = h + self.position_embeddings(positions).expand_as(h)
        h = self.dropout(h)

        attn_mask = None
        if self.causal:
            attn_mask = torch.full((len(x), len(x)), -float('Inf'), device=h.device, dtype=h.dtype)
            attn_mask = torch.triu(attn_mask, diagonal = 1)
        for layer_norm_1, attention, layer_norm_2, feed_forward in zip(self.layer_norms_1, self.attentions,
                                                                       self.layer_norms_2, self.feed_forwards):
            h = layer_norm_1(h)
            x, _ = attention(h, h, h, attn_mask = attn_mask, need_weights = False, key_padding_mask = padding_mask)
            x = self.dropout(x)
            h = x + h

            h = layer_norm_2(h)
            x = feed_forward(h)
            x = self.dropout(x)
            h = x + h
        return h


class TransformerWithLMHead(nn.Module):
    def __init__(self, embed_dim, 
                 hidden_dim, 
                 num_embeddings,
                 num_max_positions, 
                 num_heads, 
                 num_layers,
                 dropout, 
                 sinusoidal_embeddings, 
                 mlm,
                 initializer_range):
      
        """ Transformer with a language modeling head on top (tied weights) """

        super().__init__()
        self.transformer = Transformer(embed_dim, hidden_dim, num_embeddings,
                                       num_max_positions, num_heads, num_layers,
                                       dropout, sinusoidal_embeddings, causal=not mlm)
        self.lm_head = nn.Linear(embed_dim, num_embeddings, bias=False)
        self.apply(self.init_weights)
        self.tie_weights()

    def tie_weights(self):
        self.lm_head.weight = self.transformer.tokens_embeddings.weight

    def init_weights(self, module):
        """ initialize weights - note that nn.MultiheadAttention is already initalized by PyTorch (xavier_uniform) """
        if isinstance(module, (nn.Linear, nn.Embedding, nn.LayerNorm)):
            module.weight.data.normal_(mean=0.0, std=initializer_range)
        if isinstance(module, (nn.Linear, nn.LayerNorm)) and module.bias is not None:
            module.bias.data.zero_()

    def forward(self, x, labels=None, padding_mask=None):
        """ Input has shape [seq length, batch] """
        hidden_states = self.transformer(x, padding_mask)
        logits = self.lm_head(hidden_states)

        if labels is not None:
            shift_logits = logits[:-1] if self.transformer.causal else logits
            shift_labels = labels[1:] if self.transformer.causal else labels
            loss_fct = nn.CrossEntropyLoss(ignore_index=-1)
            loss = loss_fct(shift_logits.view(-1, shift_logits.size(-1)), shift_labels.view(-1))
            return logits, loss

        return logits

In [11]:
embed_dim = 128
hidden_dim = 128
num_embeddings = len(TEXT.vocab.itos) + 1
num_max_positions = bptt
num_heads = 8
num_layers = 6
dropout = 0.1
sinusoidal_embeddings = True
mlm = True
causal = not mlm
initializer_range = 0.02
lr = 2.5e-4
# lr = 1
weight_decay = 0.0
gradient_accumulation_steps = 1
max_norm = 0.25
log_interval = 20

model = TransformerWithLMHead(embed_dim, 
                 hidden_dim, 
                 num_embeddings,
                 num_max_positions, 
                 num_heads, 
                 num_layers,
                 dropout, 
                 sinusoidal_embeddings, 
                 mlm,
                 initializer_range)

In [12]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'The model has {count_parameters(model):,} trainable parameters')

The model has 4,293,120 trainable parameters


In [13]:
import torch.optim as optim

optimizer = optim.Adam(model.parameters(), lr = lr, weight_decay = weight_decay)

model=model.to(device)


## 3. MLM Fine-Tuning Causal Loop
We train the model, optimizing predictions over masked elements only.


In [15]:
def train(model, iterator):
    clip = 0.25
    total_loss = 0
    
    model.train()
            
    for k, batch in enumerate(iterator):
        inputs = batch.text.contiguous()
        inputs, labels = mask_tokens(inputs) if mlm else (inputs, inputs)

        inputs = inputs.to(device)

        optimizer.zero_grad()

        logits, loss = model(inputs, labels=labels)

        loss = loss / gradient_accumulation_steps
                      
        loss.backward()
        
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm)

        optimizer.step()
        
        total_loss += loss.item()
        
        if k % log_interval == 0 and k > 0:
            cur_loss = total_loss / log_interval
            print('| epoch {:3d} | {:5d}/{:5d} batches | lr {} | '
                    'loss {:5.2f} | ppl {:8.2f}'.format(epoch, k, len(iterator), lr, cur_loss, math.exp(cur_loss)))
            total_loss = 0
        # return loss.item()

loss_fct = nn.CrossEntropyLoss(ignore_index=-1)


def evaluate(model, iterator):
    total_loss = 0
    
    model.eval()
        
    with torch.no_grad():
    
        for batch in iterator:
            inputs = batch.text.contiguous()
            inputs, labels = mask_tokens(inputs) if mlm else (inputs, inputs)

            inputs = inputs.to(device)

            logits = model(inputs)
            
            shift_logits = logits[:-1] if not mlm else logits

            shift_labels = labels[1:] if not mlm else labels

            # total_loss += len(data) * loss
            
            return loss_fct(shift_logits.view(-1, logits.size(-1)), shift_labels.view(-1))


In [16]:
import time

def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

In [17]:
import math

N_EPOCHS = 100

best_valid_loss = float('inf')
counter = 0
patience = 5

for epoch in range(N_EPOCHS):

    start_time = time.time()
    
    train(model, train_iter)
    valid_loss = evaluate(model, valid_iter)
    
    end_time = time.time()

    epoch_mins, epoch_secs = epoch_time(start_time, end_time)

    print(f'Epoch: {epoch+1:02} | Epoch Time: {epoch_mins}m {epoch_secs}s')
    print(f'\t Val. Loss: {valid_loss:.3f} |  Val. PPL: {math.exp(valid_loss):.2f}')

    # if valid_loss < best_valid_loss:
    #     best_valid_loss = valid_loss
    #     #torch.save(model.state_dict(), 'tut2-model.pt')
    #     counter = 0 
    # else:
    #     lr /= 4.0
    #     counter += 1
    #     if counter >= patience:
    #         break


| epoch   0 |    20/  224 batches | lr 0.00025 | loss 10.77 | ppl 47383.38
| epoch   0 |    40/  224 batches | lr 0.00025 | loss 10.16 | ppl 25851.93
| epoch   0 |    60/  224 batches | lr 0.00025 | loss  9.82 | ppl 18466.12
| epoch   0 |    80/  224 batches | lr 0.00025 | loss  9.01 | ppl  8208.56
| epoch   0 |   100/  224 batches | lr 0.00025 | loss  7.76 | ppl  2338.58
| epoch   0 |   120/  224 batches | lr 0.00025 | loss  6.96 | ppl  1058.21
| epoch   0 |   140/  224 batches | lr 0.00025 | loss  6.84 | ppl   938.31
| epoch   0 |   160/  224 batches | lr 0.00025 | loss  6.87 | ppl   962.22
| epoch   0 |   180/  224 batches | lr 0.00025 | loss  6.81 | ppl   905.05
| epoch   0 |   200/  224 batches | lr 0.00025 | loss  6.80 | ppl   896.70
| epoch   0 |   220/  224 batches | lr 0.00025 | loss  6.83 | ppl   925.62
Epoch: 01 | Epoch Time: 1m 23s
	 Val. Loss: 6.234 |  Val. PPL: 509.86
| epoch   1 |    20/  224 batches | lr 0.00025 | loss  7.13 | ppl  1248.80
| epoch   1 |    40/  224 batc

KeyboardInterrupt: ignored